# Fit the weighted fixed-effects model on index change

Fit the weighted two-way fixed-effects difference-in-differences model to
site-level mean changes from `outputs/change_ci/`, per
`configs/change_did.yaml`: the main model at the primary τ, both placebo
tests, and the sensitivity reruns, for every configured index and month.
Results are saved under `outputs/change_did/<INDEX>/`. For the method, see the
[handoff brief](../docs/working-docs/Handoff-Weighted-Fixed-Effects-Model-on-Index-Change.md).

In [1]:
from pathlib import Path

import pandas as pd

from change_did import (
    ChangeDidAnalysis,
    DidAnalysisConfig,
    flagged_se_report,
    load_change_results,
)

ROOT = Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run notebooks from the notebooks/ folder"
pd.set_option("display.width", 160)

## Analysis configuration

In [2]:
config = DidAnalysisConfig.from_yaml(
    ROOT / "configs" / "change_did.yaml", project_root=ROOT
)
config.to_dict()

{'input_paths': ['/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/NDVI/results.csv',
  '/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/GNDVI/results.csv',
  '/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/SAVI/results.csv',
  '/home/tshihadah/code/wadi-gaza-research/outputs/change_ci/NDRE/results.csv'],
 'output_dir': '/home/tshihadah/code/wadi-gaza-research/outputs/change_did',
 'indices': ('NDVI', 'GNDVI', 'SAVI', 'NDRE'),
 'primary_index': 'NDVI',
 'months': (1, 5, 9, 11),
 'supplementary_months': (1,),
 'affected_site': 'WadiGaza',
 'event_pairs': [[2023, 2024], [2024, 2025]],
 'tau': 0.05,
 'tau_sensitivity': (0.02, 0.05, 0.1),
 'alpha': 0.05,
 'fill_n_eff': 5,
 'expected_flagged_primary': 22}

## Load and validate the change-CI results

Loading runs the brief's checks: every index × month × τ subset holds one row
per site × year pair with no duplicates, `year_end = year_start + 1`, `se` is
missing exactly where `flag_no_range` is true, and `mean_change` is never
missing. A failed check raises with a clear message.

In [3]:
frame = load_change_results(config)
expected = (
    len(config.indices) * len(config.months) * len(config.tau_sensitivity)
    * frame["study_area"].nunique()
    * frame[["year_start", "year_end"]].drop_duplicates().shape[0]
)
if len(frame) != expected:
    raise ValueError(f"Expected {expected} input rows, loaded {len(frame)}")
print(f"Loaded {len(frame)} rows for {frame['study_area'].nunique()} sites")
frame.head()

Loaded 2016 rows for 7 sites


,study_area,index,month,year_start,year_end,tau,is_primary_tau,engine,n,mean_change,...,se,ci_low,ci_high,naive_se,naive_ci_low,naive_ci_high,ci_excludes_zero,flag_no_range,flag_low_n_eff,error
0,AlAhrashProtectorate,NDVI,1,2019,2020,0.02,False,fft,33848,0.025359,...,0.005149,0.015121,0.035598,0.000259,0.024852,0.025867,True,False,False,NaN
1,AlAhrashProtectorate,NDVI,1,2019,2020,0.05,True,fft,33848,0.025359,...,0.004832,0.015769,0.034950,0.000259,0.024852,0.025867,True,False,False,NaN
2,AlAhrashProtectorate,NDVI,1,2019,2020,0.10,False,fft,33848,0.025359,...,0.004379,0.016687,0.034032,0.000259,0.024852,0.025867,True,False,False,NaN
3,AlAhrashProtectorate,NDVI,1,2020,2021,0.02,False,fft,33848,-0.019612,...,NaN,NaN,NaN,0.000321,-0.020241,-0.018983,False,True,False,NaN
4,AlAhrashProtectorate,NDVI,1,2020,2021,0.05,True,fft,33848,-0.019612,...,0.008229,-0.036136,-0.003088,0.000321,-0.020241,-0.018983,True,False,False,NaN


## Flagged (missing-SE) rows at the primary τ

The brief expects 22 flagged NDVI rows at τ = 0.05 (5 each in May, September
and November; 7 in January); a different NDVI count logs a warning, not an
error. Counts for the other indices are reported for the record.

In [4]:
analysis = ChangeDidAnalysis(config)
flagged = flagged_se_report(frame, config)
flagged

,index,month_01,month_05,month_09,month_11,total
0,NDVI,7,5,5,5,22
1,GNDVI,7,6,3,5,21
2,SAVI,6,5,4,4,19
3,NDRE,9,5,7,4,25


## Run and save

Each index × month runs the primary specification plus the sensitivity
variants (τ ∈ {0.02, 0.05, 0.10}, exclude-flagged, n_eff = 5 fill), and the
primary specification adds both placebo tests.

In [5]:
results = analysis.run(frame)
analysis.save(results)
print(
    f"Saved {len(results.main)} main rows, {len(results.placebo_space)} placebo-space rows, "
    f"and {len(results.placebo_spacetime)} placebo-space-and-time rows under {config.output_dir}"
)
results.main.groupby(["index", "variant"]).size().unstack()

Dropped indicator d_persist for NDVI month 9 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for NDVI month 11 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for SAVI month 9 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for NDRE month 9 variant exclude_flagged: its cell is absent after SE handling


Dropped indicator d_persist for NDRE month 11 variant exclude_flagged: its cell is absent after SE handling


Saved 155 main rows, 112 placebo-space rows, and 560 placebo-space-and-time rows under /home/tshihadah/code/wadi-gaza-research/outputs/change_did


variant,exclude_flagged,n_eff_fill,primary,tau_0.02,tau_0.1
index,,,,,
GNDVI,8,8,8,8,8
NDRE,6,8,8,8,8
NDVI,6,8,8,8,8
SAVI,7,8,8,8,8


## Primary-specification estimates

One row per index × month × term at the primary τ with the largest-SE fill.
`d_event` is the estimate of interest (β, first event pair) and `d_persist`
is the persistence estimate (β′, second event pair). A clipped σ_u² means the
measurement SEs exceed the total scatter.

In [6]:
primary = results.main.loc[results.main["variant"] == "primary"]
display_columns = [
    "index", "role", "month", "supplementary", "term", "estimate",
    "ci_low", "ci_high", "p_value", "sigma_u", "sigma_u_clipped", "df", "n_filled",
]
primary[display_columns].sort_values(["index", "month", "term"]).reset_index(drop=True)

,index,role,month,supplementary,term,estimate,ci_low,ci_high,p_value,sigma_u,sigma_u_clipped,df,n_filled
0,GNDVI,robustness,1,True,d_event,-0.080461,-0.191892,0.030971,0.150281,0.041253,False,28,7
1,GNDVI,robustness,1,True,d_persist,-0.037984,-0.147111,0.071144,0.481756,0.041253,False,28,7
2,GNDVI,robustness,5,False,d_event,-0.059972,-0.109334,-0.010611,0.019035,0.018319,False,28,6
3,GNDVI,robustness,5,False,d_persist,-0.013833,-0.064258,0.036592,0.578637,0.018319,False,28,6
4,GNDVI,robustness,9,False,d_event,-0.073426,-0.117691,-0.029160,0.002054,0.013239,False,28,3
5,GNDVI,robustness,9,False,d_persist,-0.024964,-0.066644,0.016717,0.230096,0.013239,False,28,3
6,GNDVI,robustness,11,False,d_event,-0.099597,-0.153265,-0.045928,0.000714,0.017155,False,28,5
7,GNDVI,robustness,11,False,d_persist,0.004939,-0.040996,0.050875,0.827270,0.017155,False,28,5
8,NDRE,robustness,1,True,d_event,-0.069871,-0.183329,0.043587,0.217548,0.040776,False,28,9
9,NDRE,robustness,1,True,d_persist,-0.090185,-0.200537,0.020168,0.105258,0.040776,False,28,9


## Placebo p-values per index and month

In [7]:
space_p = results.placebo_space.groupby(["index", "month"])["p_value"].first()
spacetime_p = results.placebo_spacetime.groupby(["index", "month"])["p_value"].first()
ratio = results.placebo_space.groupby(["index", "month"])["ratio_to_largest_placebo"].first()
pd.DataFrame({
    "placebo_space_p": space_p,
    "ratio_to_largest_placebo": ratio,
    "placebo_spacetime_p": spacetime_p,
}).reset_index()

,index,month,placebo_space_p,ratio_to_largest_placebo,placebo_spacetime_p
0,GNDVI,1,0.142857,0.791989,0.057143
1,GNDVI,5,0.142857,2.505064,0.028571
2,GNDVI,9,0.142857,4.666235,0.028571
3,GNDVI,11,0.142857,2.701028,0.028571
4,NDRE,1,0.142857,0.723797,0.085714
5,NDRE,5,0.142857,3.280757,0.028571
6,NDRE,9,0.142857,6.827897,0.028571
7,NDRE,11,0.142857,12.730437,0.028571
8,NDVI,1,0.142857,0.720516,0.085714
9,NDVI,5,0.142857,3.133183,0.028571


## Validate against the brief's NDVI targets

`tests/test_did_validation.py` reproduces the handoff brief's reference run to
within ±0.001 (main model, both placebos, and every sensitivity variant). Run
it here as a final gate; a mismatch means the design, the fill, or the
σ_u² step differs from the brief.

In [8]:
import subprocess
import sys

check = subprocess.run(
    [sys.executable, "-m", "pytest", str(ROOT / "tests" / "test_did_validation.py"), "-q"],
    capture_output=True, text=True, cwd=ROOT,
)
print(check.stdout[-2000:])
if check.returncode != 0:
    print(check.stderr[-2000:])
    raise RuntimeError("Validation targets failed; outputs do not match the brief")

.............................                                            [100%]
29 passed in 0.37s

